# Panel F MM001

evaluate dna_r10.4.1_e8.2_400bps_sup@_4.1.0 finetuned model on MM001 Fiber-Seq data

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib
matplotlib.rcParams['pdf.fonttype'] = 42

In [ ]:
# change paths as appropriate
model_410_file='/data/basecalled/20230816_dorado_mod_MM001/pod5/dna_r10.4.1_e8.2_400bps_sup@_4.1.0_basecalls_summary.tsv.gz'
model_finetuned_file='/data/basecalled/20230816_dorado_mod_MM001/pod5/retrained_basecalls_summary.tsv.gz'

In [ ]:
model_410 = pd.read_csv(model_410_file, delimiter='\t')

In [ ]:
model_finetuned = pd.read_csv(model_finetuned_file, delimiter='\t')

In [ ]:
font_size = 18

plt.figure(figsize=(8, 6))

# plot only read with alignment identity >0.8
model_finetuned_subset = model_finetuned[model_finetuned['alignment_identity'] > 0.8]
model_410_subset = model_410[model_410['alignment_identity'] > 0.8]

sns.kdeplot(
    model_finetuned_subset['alignment_identity'],
    label='sup_4.1.0-finetuned',
    color='#1A2A4F',
    linewidth=2,
    fill=True,
    alpha=0.6
)

sns.kdeplot(
    model_410_subset['alignment_identity'],
    label='sup_4.1.0',
    color='#F7A5A5',
    linewidth=2,
    fill=True,
    alpha=0.6
)

plt.xlabel('Read alignment Identity', fontsize=font_size)
plt.ylabel('Density', fontsize=font_size)
plt.title('Alignment Identity Distribution MM001 SQK-LSK114 4kHz data', fontsize=font_size)


plt.legend(
    title='Model',
    loc='upper left',
    fontsize=font_size,
    title_fontsize=font_size,
    frameon=False
)


plt.xlim(0.8, 1)
plt.xticks(fontsize=font_size)
plt.yticks(fontsize=font_size)
plt.tight_layout()
plt.savefig('figures/MM001_SQKLSK114.pdf', format='pdf', bbox_inches='tight')
plt.show()

# Panel E: HG001 evaluation


### get the GIAB benchmark calls and regions
```
wget https://ftp-trace.ncbi.nlm.nih.gov/giab/ftp/release/NA12878_HG001/latest/GRCh38/HG001_GRCh38_1_22_v4.2.1_benchmark.vcf.gz
wget https://ftp-trace.ncbi.nlm.nih.gov/giab/ftp/release/NA12878_HG001/latest/GRCh38/HG001_GRCh38_1_22_v4.2.1_benchmark.vcf.gz.tbi
wget https://ftp-trace.ncbi.nlm.nih.gov/giab/ftp/release/NA12878_HG001/latest/GRCh38/HG001_GRCh38_1_22_v4.2.1_benchmark.bed
```
### get singularity image
```
singularity pull docker://jmcdani20/hap.py:v0.3.12
```
### run happy
```
export HGREF=/references/GRCh38.alt-masked-V2-noALT/index/minimap2-ont/Homo_sapiens_assembly38_masked_noALT.fasta
singularity exec -B /data/leuven/ -B /staging/leuven/ \
/data/leuven/software/biomed/singularity_images/images_jonas/hap.py_v0.3.12.sif /opt/hap.py/bin/hap.py \
/ASA/analysis/20221130_clair3_test_AP/GIAB_calls/HG001/HG001_GRCh38_1_22_v4.2.1_benchmark.vcf.gz \
/ASA/analysis/20221130_clair3_test_AP/NA12878_fiber_HG38/merge_output.vcf.gz \
-f /ASA/analysis/20221130_clair3_test_AP/GIAB_calls/HG001/HG001_GRCh38_1_22_v4.2.1_benchmark.bed \
-r /references/GRCh38.alt-masked-V2-noALT/index/minimap2-ont/Homo_sapiens_assembly38_masked_noALT.fasta \
-o /ASA/analysis/20221130_clair3_test_AP/NA12878_fiber_HG38/2025_happy/2025_happy.output --pass-only --engine=vcfeval 
```

In [ ]:
vals = pd.read_csv('/NA12878_fiber_HG38/2025_happy/2025_happy.output.summary.csv') # change as appropriate
vals = vals[vals['Filter'] == 'PASS']

In [ ]:
vals = vals.rename({'METRIC.Recall': 'Recall',
             'METRIC.Precision': 'Precision',
             'METRIC.F1_Score': 'F1_score'
            }, axis='columns')

In [ ]:
metrics = ['Recall', 'Precision', 'F1_score']

custom_palette = {'SNP': '#A5BFCC', 'INDEL': '#F7A5A5'}

df_melted = vals.melt(
    id_vars='Type',
    value_vars=metrics,
    var_name='Metric',
    value_name='Value'
)

font_size = 18

plt.figure(figsize=(10, 6))
ax = sns.barplot(
    data=df_melted,
    x='Metric',
    y='Value',
    hue='Type',
    edgecolor='black',
    palette=custom_palette
)

for container in ax.containers:
    for bar in container:
        height = bar.get_height()
        if not pd.isna(height):
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                height,
                f"{height:.4f}",
                ha='center',
                va='bottom',
                fontsize=font_size,
                color='black'
            )

ax.set_title('NA12878 SQK-LSK110', fontsize=font_size)
ax.set_xlabel('', fontsize=font_size)
ax.set_ylabel('', fontsize=font_size)
ax.tick_params(axis='both', labelsize=font_size)

ax.legend(
    title='Type',
    bbox_to_anchor=(1.05, 1),
    loc='upper left',
    frameon=False,
    fontsize=font_size,
    title_fontsize=font_size
)

plt.tight_layout()
plt.savefig('/figures/barplot_fiberseq_call_qc_r9.pdf', format='pdf', bbox_inches='tight')
plt.show()